# Core types and money

**Purpose:** Learn finstack-quant's typed primitives for currencies, cash amounts, rates, identifiers, metadata, and library configuration.

*No prerequisites — this is the first notebook in the curriculum.*


## Why financial type safety matters

In production finance code, raw floats and strings are easy to misuse: adding dollars to euros, confusing percent with decimal, or treating identifiers as interchangeable. finstack wraps these concepts in small, explicit types so invalid combinations fail early and APIs stay self-documenting.


## `Currency` — ISO-4217 currency codes

Immutable currency values with alphabetic and numeric ISO codes, decimals for formatting, and JSON helpers.


In [ ]:
from finstack_quant.core.currency import Currency, USD, EUR, GBP, JPY

c = Currency("USD")
c_num = Currency.from_numeric(840)
print("from string:", c.code, c.numeric, c.decimals)
print("from numeric:", c_num.code, c_num.numeric)

j = c.to_json()
print("to_json:", j)
print("from_json:", Currency.from_json(j).code)

print("constants:", USD.code, EUR.code, GBP.code, JPY.code)
print("compare Currency:", USD == EUR, USD != EUR)
print("ordering (alphabetic ISO codes):", EUR < USD, GBP > EUR)
print("compare with str:", Currency("USD") == "USD")


## `Money` — amounts tagged with a currency

Construct with a string or `Currency`, format for display, serialize, and do arithmetic — mixing currencies raises `ValueError`.


In [ ]:
from finstack_quant.core.currency import Currency
from finstack_quant.core.money import Money

m1 = Money(100.0, "USD")
m2 = Money(100.0, Currency("USD"))
z = Money.zero("USD")
print("constructors equal:", m1 == m2)
print("zero:", z.amount, z.currency.code)
print("amount / currency:", m1.amount, m1.currency)
print("format default:", m1.format())
print("format 4dp:", m1.format(decimals=4))

js = m1.to_json()
print("json round-trip:", Money.from_json(js))
t = m1.to_tuple()
print("tuple:", t, "from_tuple:", Money.from_tuple(t))

x = m1 + Money(50.0, "USD")
y = x - Money(25.0, "USD")
print("add/sub:", x, y)
print("mul/div/neg:", m1 * 1.5, m1 / 2.0, -m1)

try:
    _ = Money(1.0, "USD") + Money(1.0, "EUR")
except ValueError as e:
    print("cross-currency add:", type(e).__name__, str(e))


## `Rate` — decimal, percent, and basis points

A single rate type with conversions and safe arithmetic.


In [ ]:
from finstack_quant.core.types import Rate

r = Rate(0.05)
r_pct = Rate.from_percent(5.0)
r_bp = Rate.from_bp(500)
print("as_decimal:", r.as_decimal)
print("as_percent:", r.as_percent)
print("as_bp:", r.as_bp)
print("from_percent matches:", r == r_pct)
print("from_bp matches:", r == r_bp)
print("ZERO:", Rate.ZERO.as_decimal)

a = Rate(0.03) + Rate(0.02)
b = Rate(0.10) - Rate(0.02)
c = Rate(0.04) * 2.0
d = Rate(0.10) / 2.0
print("arithmetic:", a.as_decimal, b.as_decimal, c.as_decimal, d.as_decimal)


## `Bps` — basis points (1 bp = 0.01%)

Integer-rounded basis-point amounts with decimal conversion.


In [ ]:
from finstack_quant.core.types import Bps

b = Bps(250)
print("as_decimal:", b.as_decimal)
print("as_bp:", b.as_bp)
print("ZERO:", Bps.ZERO.as_bp)
print("arithmetic:", (Bps(100) + Bps(50)).as_bp, (Bps(300) - Bps(50)).as_bp)


## `Percentage` — human percent values

Stores values like `12.5` meaning 12.5%.


In [ ]:
from finstack_quant.core.types import Percentage

p = Percentage(12.5)
print("as_decimal:", p.as_decimal)
print("as_percent:", p.as_percent)
print("ZERO:", Percentage.ZERO.as_percent)


## `CreditRating` — notch-preserving rating categories

Use class constants or parse S&P/Fitch-style and Moody's strings. Notches are preserved exactly: for example, both `BBB+` and `Baa1` map to `CreditRating.BBB_PLUS`, whose `name` is `"BBB+"` and whose `warf` exposes the notch-specific Moody's factor.


In [ ]:
from finstack_quant.core.types import CreditRating

print("AAA:", CreditRating.AAA.name)
r1 = CreditRating.from_name("BBB")
r2 = CreditRating.from_name("BBB+")
r3 = CreditRating.from_name("Baa1")
print("from_name BBB:", r1.name)
print("from_name BBB+:", r2.name)
print("from_name Baa1:", r3.name)


## Scorecard scales via `rating_scales` registry

The `finstack_quant.core.rating_scales` module exposes the versioned registry of agency-style scorecard scales (S&P, Moody's, Fitch, ...). This is distinct from the normalized `CreditRating` enum buckets shown above.


In [ ]:
from finstack_quant.core.rating_scales import (
    embedded_registry,
    registry_from_config,
)
from finstack_quant.core.config import FinstackConfig

reg = embedded_registry()
print("default scale:", reg.default_scale_id())
print("known 'sp':", reg.is_known_rating_scale("sp"))
print("known 'nope':", reg.is_known_rating_scale("nope"))

sp = reg.rating_scale("sp")
print("SP scale name:", sp.scale_name, "#levels:", len(sp))
for r in list(sp.ratings)[:4]:
    print(f"  {r.name:>4s} score={r.score} min={r.min_score}")

# From a (default) config — falls back to embedded when no extension override
reg_cfg = registry_from_config(FinstackConfig())
print("from default config:", reg_cfg.default_scale_id())


## `CurveId` and `InstrumentId` — stable string identifiers

Lightweight wrappers for curve and instrument keys used across market data and instruments.


In [ ]:
from finstack_quant.core.types import CurveId, InstrumentId

cid = CurveId("USD-OIS")
iid = InstrumentId("BOND-001")
print("CurveId:", cid.as_str())
print("InstrumentId:", iid.as_str())


## `Attributes` — string metadata bag

Attach key/value tags (all strings) to instruments or structures.


In [ ]:
from finstack_quant.core.types import Attributes

attrs = Attributes()
attrs.set_meta("sector", "Technology")
attrs.set_meta("rating", "BBB")
print("get sector:", attrs.get_meta("sector"))
print("contains sector:", attrs.contains_meta_key("sector"))
print("keys:", attrs.keys())
print("len:", len(attrs))


## `RoundingMode`, `ToleranceConfig`, and `FinstackConfig`

Control rounding semantics, numerical tolerances, and per-currency output scales.


In [ ]:
from finstack_quant.core.config import RoundingMode, ToleranceConfig, FinstackConfig

rm = RoundingMode.BANKERS
rm2 = RoundingMode.from_name("bankers")
print("rounding modes equal:", rm == rm2)

tc0 = ToleranceConfig()
tc1 = ToleranceConfig(rate_epsilon=1e-10)
print("default rate_epsilon:", tc0.rate_epsilon)
print("custom rate_epsilon:", tc1.rate_epsilon)

cfg0 = FinstackConfig()
cfg1 = FinstackConfig(rounding_mode=RoundingMode.BANKERS)
print(
    "default output_scale USD/JPY:",
    cfg0.output_scale("USD"),
    cfg0.output_scale("JPY"),
)
print("with bankers rounding repr:", cfg1)


## Mini-example: multi-currency cash position tracker

Track balances in three currencies, apply deposits and fees, print formatted balances, and show cross-currency mistakes raising errors.


In [ ]:
from finstack_quant.core.money import Money

positions = {
    "USD": Money.zero("USD"),
    "EUR": Money.zero("EUR"),
    "JPY": Money.zero("JPY"),
}

def deposit(ccy: str, amt: float) -> None:
    positions[ccy] = positions[ccy] + Money(amt, ccy)

def fee(ccy: str, amt: float) -> None:
    positions[ccy] = positions[ccy] - Money(amt, ccy)

deposit("USD", 1_000_000.0)
deposit("EUR", 250_000.0)
deposit("JPY", 50_000_000.0)
fee("USD", 1_250.50)
fee("EUR", 300.0)

print("--- Balances ---")
for ccy in ("USD", "EUR", "JPY"):
    print(positions[ccy].format())

usd_net = positions["USD"] + Money(10_000.0, "USD")
print("USD after inflow:", usd_net.format())

print("--- Cross-currency guard ---")
try:
    _ = positions["USD"] + positions["EUR"]
except ValueError as e:
    print("blocked:", e)


## Takeaways

- **`Currency` / `Money`** encode ISO currencies and tagged amounts; arithmetic refuses currency mismatches.
- **`Rate`, `Bps`, `Percentage`** give consistent conversions between decimal, percent, and basis-point views.
- **`CreditRating`, `CurveId`, `InstrumentId`, `Attributes`** standardize ratings, IDs, and metadata strings.
- **`FinstackConfig`** ties rounding and tolerances to library-wide behavior, including output precision by currency.

**Next:** Continue with `01_foundations/dates_calendars_schedules.ipynb`, or jump to `02_pricing/pricing_fundamentals.ipynb` once these primitives feel familiar.


## Analyst program: money, FX and a reproducible result

Money arithmetic retains decimal amounts. Pricing records its own numeric and rounding policy; the current native pricer uses f64 calculations with a rounding stamp. A currency mismatch raises the public `ValueError`. Compare financial outputs when testing replay: timestamps describe separate executions.

In [ ]:
from datetime import date
import json
from decimal import Decimal
from finstack_quant.core.money import Money
from finstack_quant.core.currency import Currency
from finstack_quant.core.config import FinstackConfig, RoundingMode
from finstack_quant.core.market_data import FxMatrix, fx_market_pair, fx_pip_size, invert_fx_rate
from finstack_quant.valuations.instruments import price_instrument
from _shared.analyst_book import book_spec, build_book, build_market, instruments
AS_OF = date(2025, 1, 15)

assert 0.1 + 0.2 != 0.3
cash = Money(Decimal('0.1'), 'USD') + Money(Decimal('0.2'), 'USD')
assert cash.amount_decimal == Decimal('0.3')
mixed_currency_error = None
try:
    cash + Money(Decimal('0.3'), 'EUR')
except ValueError as error:
    mixed_currency_error = str(error)
assert mixed_currency_error is not None and 'currency' in mixed_currency_error.lower()
print('Mixed-currency addition rejected:', mixed_currency_error)
fx = FxMatrix()
fx.set_quote(Currency('EUR'), Currency('USD'), 1.08)
fx.set_quote(Currency('USD'), Currency('JPY'), 150.0)
cross = fx.rate('EUR', 'JPY', AS_OF)
assert abs(cross.rate - 162.0) < 1e-12
assert abs(invert_fx_rate(cross.rate) * cross.rate - 1) < 1e-12
assert [currency.code for currency in fx_market_pair('USD', 'EUR')] == ['EUR', 'USD']
assert fx_pip_size('EUR', 'JPY') == 0.01
config = FinstackConfig(rounding_mode=RoundingMode.BANKERS)
market = build_market('foundations')
bond = json.dumps(instruments('foundations')['USD-CORP'])
first = price_instrument(bond, market, AS_OF)
second = price_instrument(bond, market, AS_OF)
assert first.price == second.price
assert first.meta['rounding']['mode'] == json.loads(config.to_json())['rounding']['mode']
assert first.meta['numeric_mode'] == 'f64'
assert config.output_scale('USD') == 2 and config.output_scale('JPY') == 0
foundation_spec = book_spec('foundations')
holdings = []
for position in foundation_spec['positions']:
    terms = position['instrument_spec']['spec']
    holdings.append({
        'position_id': position['position_id'],
        'instrument_type': position['instrument_spec']['type'],
        'quantity': position['quantity'],
        'currency': terms['notional']['currency'],
        'contract_notional': terms['notional']['amount'],
    })
print(json.dumps({'as_of': AS_OF.isoformat(), 'stage': 'foundations', 'holdings': holdings}, indent=2))
assert foundation_spec['as_of'] == AS_OF.isoformat()
assert len(build_book('foundations')) == len(holdings) == 4

## Units and banker rounding

In [ ]:
from decimal import Decimal, ROUND_HALF_EVEN
from finstack_quant.core.types import Rate, Bps, Percentage
from finstack_quant.core.config import FinstackConfig
assert Rate.from_percent(5).as_decimal == 0.05
assert Bps(250).as_decimal == 0.025
assert Percentage(5).as_decimal == 0.05
rounded = [Decimal(x).quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN) for x in ('1.005','1.015')]
assert rounded == [Decimal('1.00'), Decimal('1.02')]
print({'decimal_rate': Rate.from_percent(5).as_decimal, '250bp': Bps(250).as_decimal,
       'banker_rounding': rounded, 'currency_output_scales': {ccy: FinstackConfig().output_scale(ccy) for ccy in ('USD','EUR','JPY')}})


## EURJPY triangulation

In [ ]:
from datetime import date
from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import FxMatrix, fx_pip_size, invert_fx_rate
AS_OF = date(2025, 1, 15)
crosses=FxMatrix()
crosses.set_quote(Currency('EUR'),Currency('USD'),1.08)
crosses.set_quote(Currency('USD'),Currency('JPY'),150.0)
eur_jpy=crosses.rate('EUR','JPY',AS_OF).rate
assert abs(eur_jpy-162.0)<1e-12 and fx_pip_size('EUR','JPY')==0.01
assert abs(invert_fx_rate(eur_jpy)*eur_jpy-1)<1e-12
print({'JPY_per_EUR':eur_jpy,'JPY_pip':fx_pip_size('EUR','JPY'),'EUR_per_JPY':invert_fx_rate(eur_jpy)})


## Accumulating small cashflows

In [ ]:
from decimal import Decimal
from finstack_quant.core.money import Money
binary_total=0.0
for _ in range(10000):binary_total += 0.01
decimal_total=sum((Money(Decimal('0.01'),'USD') for _ in range(10000)),Money(0,'USD'))
assert binary_total != 100.0
assert decimal_total.amount_decimal == Decimal('100.00')
print({'binary_total':repr(binary_total),'binary_residual':binary_total-100.0,'money_total':str(decimal_total.amount_decimal)})


## Independent GBP cash extension

In [ ]:
from datetime import date
import copy,json
from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import DiscountCurve,FxMatrix
from finstack_quant.portfolio import Portfolio,value_portfolio
from _shared.analyst_book import book_spec, build_market
AS_OF = date(2025, 1, 15)
original=book_spec('foundations'); original_wire=json.dumps(original,sort_keys=True)
extended=copy.deepcopy(original)
gbp=copy.deepcopy(next(p for p in extended['positions'] if p['position_id']=='EUR-CASH'))
gbp.update(position_id='GBP-CASH',instrument_id='GBP-CASH')
gbp['instrument_spec']['spec'].update(id='GBP-CASH',notional={'amount':'100000','currency':'GBP'},discount_curve_id='GBP-OIS')
extended['positions'].append(gbp)
gbp_market=build_market('foundations').insert(DiscountCurve.flat('GBP-OIS',AS_OF,0.04))
quotes=FxMatrix();quotes.set_quote(Currency('EUR'),Currency('USD'),1.08);quotes.set_quote(Currency('GBP'),Currency('USD'),1.25)
gbp_market.insert_fx(quotes)
valuation=value_portfolio(Portfolio.from_spec(json.dumps(extended)),gbp_market,metrics=[])
wire=json.loads(valuation.to_json());holding=wire['position_values']['GBP-CASH']
assert len(extended['positions'])==5 and len(original['positions'])==4
assert json.dumps(original,sort_keys=True)==original_wire
assert abs(float(holding['value_base']['amount'])-1.25*float(holding['value_native']['amount']))<0.01
print([{'id':p['position_id'],'currency':p['instrument_spec']['spec']['notional']['currency']} for p in extended['positions']])
print({'GBP_value':holding['value_native'],'USD_value':holding['value_base']})
